# Predicting Student Admissions with Neural Networks in PyTorch
In this notebook we predict student admissions to graduate school at UCLA from three pieces of data:
- GRE scores (test)
- GPA scores (grades)
- Class rank (1-4)

The dataset originally came from http://www.ats.ucla.edu/. This is the same problem as the NumPy version in `student-admissions/`, now built with [PyTorch](https://pytorch.org/), which handles the gradients for us.

## Loading the data
We use Pandas and NumPy to load and prepare the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

data = pd.read_csv('../student-admissions/student_data.csv')
data[:10]

## Plotting the data
First let's plot our data. To have a 2D plot, we ignore the rank for now.

In [ ]:
def plot_points(data):
    X = np.array(data[["gre", "gpa"]])
    y = np.array(data["admit"])
    admitted = X[y == 1]
    rejected = X[y == 0]
    plt.scatter(rejected[:, 0], rejected[:, 1], s=25, color='red', edgecolor='k')
    plt.scatter(admitted[:, 0], admitted[:, 1], s=25, color='cyan', edgecolor='k')
    plt.xlabel('Test (GRE)')
    plt.ylabel('Grades (GPA)')

plot_points(data)
plt.show()

Roughly, students with high grades and test scores were admitted, but the data is not cleanly separable. Maybe the rank helps? Let's make one plot per rank.

In [ ]:
for rank in [1, 2, 3, 4]:
    plot_points(data[data["rank"] == rank])
    plt.title("Rank %d" % rank)
    plt.show()

The lower the rank, the higher the acceptance rate, so we'll use the rank as an input. To do that we one-hot encode it.

## One-hot encoding the rank
We use Pandas' `get_dummies`. Passing `dtype=int` gives numeric columns instead of booleans.

In [ ]:
one_hot_data = pd.concat([data, pd.get_dummies(data['rank'], prefix='rank', dtype=int)], axis=1)
one_hot_data = one_hot_data.drop('rank', axis=1)
one_hot_data[:10]

## Scaling the data
Grades range over 1.0-4.0 while test scores range over roughly 200-800. Skewed ranges make training harder, so we scale both into 0-1.

In [ ]:
processed_data = one_hot_data.copy()
processed_data['gre'] = processed_data['gre'] / 800
processed_data['gpa'] = processed_data['gpa'] / 4.0
processed_data[:10]

## Splitting into training and testing sets
The test set is 10% of the data.

In [ ]:
sample = np.random.choice(processed_data.index, size=int(len(processed_data) * 0.9), replace=False)
train_data, test_data = processed_data.iloc[sample], processed_data.drop(sample)

print("Number of training samples is", len(train_data))
print("Number of testing samples is", len(test_data))

## Features and targets as tensors
PyTorch works with tensors. Features are `float32`; the class labels (0 = rejected, 1 = admitted) are `long` integers. We don't need to one-hot encode the output here: `nn.CrossEntropyLoss` takes the class index directly.

In [ ]:
def to_tensors(df):
    X = torch.tensor(df.drop('admit', axis=1).values, dtype=torch.float32)
    y = torch.tensor(df['admit'].values, dtype=torch.long)
    return X, y

features, targets = to_tensors(train_data)
features_test, targets_test = to_tensors(test_data)
print(features.shape, targets.shape)

## Defining the model
A network with two hidden layers and dropout. It outputs two scores (logits), one per class; the softmax is folded into the loss function.

In [ ]:
model = nn.Sequential(
    nn.Linear(6, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 2),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model

## Training the model
Each epoch goes over the training data in mini-batches. For each batch we compute the loss, backpropagate, and update the weights. Calling `model.train()` switches dropout on.

In [ ]:
loader = DataLoader(TensorDataset(features, targets), batch_size=100, shuffle=True)

epochs = 200
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 40 == 0 or epoch == epochs - 1:
        print("Epoch %3d  last batch loss %.4f" % (epoch, loss.item()))

## Scoring the model
`model.eval()` switches dropout off, and `torch.no_grad()` skips gradient tracking.

In [ ]:
def accuracy(X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

print("Training accuracy:", accuracy(features, targets))
print("Testing accuracy: ", accuracy(features_test, targets_test))

## Challenge: play with the parameters!
We made several decisions: the layer sizes, dropout, the optimizer, the learning rate, the number of epochs. Can you improve the accuracy? Things to try:
- Activation function: `nn.ReLU` or `nn.Sigmoid`
- Optimizer: `torch.optim.SGD` (with momentum), `RMSprop`, `Adam`
- Loss: `nn.CrossEntropyLoss`, or a regression loss on a single sigmoid output